# Tree-Bound LLM: Native LLM-Tree Integration

**Core idea:** The tree IS the LLM's mathematical memory.

- Cross-attention from LLM hidden states to signature embeddings
- Welford variance controls attention temperature
- Tree grows dynamically during cold start
- Autoregressive routing for multi-step problems

**Architecture:**
```
Problem → Phi-2 (frozen) → Cross-Attention to Tree → Route → Execute
                                    ↑
                          Signature embeddings (learnable)
                          Welford stats (updated at runtime)
```

**Setup:** GPU runtime (T4 minimum, A100 recommended)

In [ ]:
# Use older transformers to avoid attention bugs
!pip install transformers==4.36.0 accelerate sentencepiece --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import json
import math
from typing import Optional, Dict, List, Tuple
from dataclasses import dataclass

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Upload GSM8K data
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print("Upload train.jsonl and test.jsonl from data/gsm8k_gts/")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f'data/{name}')
    print(f"Saved: data/{name}")

## 1. Signature Storage (The Tree)

In [ ]:
@dataclass
class Signature:
    """A single signature (leaf node) in the tree."""
    id: int
    dsl: str  # The operation code, e.g., "a + b"
    description: str  # Natural language description
    
    # Welford online stats
    n: int = 0
    mean: float = 0.5  # Success rate
    m2: float = 0.0    # For variance calculation
    
    @property
    def variance(self) -> float:
        if self.n < 2:
            return 1.0  # High variance = uncertain
        return self.m2 / (self.n - 1)
    
    def update(self, success: bool):
        """Welford online update."""
        x = 1.0 if success else 0.0
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.m2 += delta * delta2


class SignatureTree:
    """Growable tree of signatures with deduplication."""
    
    def __init__(self, embedding_dim: int = 256, max_signatures: int = 500):
        self.embedding_dim = embedding_dim
        self.max_signatures = max_signatures
        
        # Signature metadata
        self.signatures: Dict[int, Signature] = {}
        
        # DSL to ID mapping for deduplication
        self.dsl_to_id: Dict[str, int] = {}
        
        # Embedding storage (will be moved to GPU)
        self.embeddings = torch.zeros(max_signatures, embedding_dim)
        self.active_mask = torch.zeros(max_signatures, dtype=torch.bool)
        self.num_active = 0
        
        # Special signatures
        self.DONE_IDX = 0  # Reserved for "done" signal
        self._init_special_signatures()
    
    def _init_special_signatures(self):
        """Initialize special signatures."""
        # DONE signature
        self.signatures[0] = Signature(
            id=0,
            dsl="DONE",
            description="Problem solved, return final answer"
        )
        self.dsl_to_id["DONE"] = 0
        self.embeddings[0] = torch.randn(self.embedding_dim) * 0.1
        self.active_mask[0] = True
        self.num_active = 1
    
    def to(self, device):
        """Move to device."""
        self.embeddings = self.embeddings.to(device)
        self.active_mask = self.active_mask.to(device)
        return self
    
    @property
    def device(self):
        """Get current device of embeddings."""
        return self.embeddings.device
    
    def has_dsl(self, dsl: str) -> bool:
        """Check if DSL already exists."""
        return dsl in self.dsl_to_id
    
    def get_id_for_dsl(self, dsl: str) -> Optional[int]:
        """Get signature ID for a DSL if it exists."""
        return self.dsl_to_id.get(dsl)
    
    def add_signature(self, embedding: torch.Tensor, dsl: str, description: str) -> Tuple[int, bool]:
        """
        Add a new signature to the tree.
        
        Returns: (signature_id, was_created)
        - If DSL exists, returns existing ID and False
        - If DSL is new, creates signature and returns new ID and True
        """
        # Ensure embedding is on the same device as our storage
        embedding = embedding.detach().to(self.device)
        
        # Check for duplicate DSL
        if dsl in self.dsl_to_id:
            existing_id = self.dsl_to_id[dsl]
            # Update embedding with running average (helps clustering)
            alpha = 0.1
            self.embeddings[existing_id] = (1 - alpha) * self.embeddings[existing_id] + alpha * embedding
            return existing_id, False
        
        if self.num_active >= self.max_signatures:
            # At capacity - find most similar existing signature
            return self._find_most_similar(embedding), False
        
        idx = self.num_active
        
        self.signatures[idx] = Signature(
            id=idx,
            dsl=dsl,
            description=description
        )
        
        self.dsl_to_id[dsl] = idx
        self.embeddings[idx] = embedding
        self.active_mask[idx] = True
        self.num_active += 1
        
        return idx, True
    
    def _find_most_similar(self, embedding: torch.Tensor) -> int:
        """Find most similar existing signature."""
        active_embs = self.embeddings[:self.num_active]
        sims = torch.cosine_similarity(embedding.unsqueeze(0), active_embs)
        return sims.argmax().item()
    
    def get_active_embeddings(self) -> torch.Tensor:
        """Get embeddings of active signatures."""
        return self.embeddings[:self.num_active]
    
    def get_variances(self) -> torch.Tensor:
        """Get Welford variances for active signatures."""
        variances = torch.tensor([
            self.signatures[i].variance 
            for i in range(self.num_active)
        ])
        return variances
    
    def update_signature(self, idx: int, success: bool):
        """Update Welford stats for a signature."""
        if idx in self.signatures:
            self.signatures[idx].update(success)
    
    def get_dsl(self, idx: int) -> str:
        """Get DSL code for a signature."""
        return self.signatures.get(idx, Signature(0, "UNKNOWN", "")).dsl
    
    def summary(self) -> str:
        """Get summary of unique DSLs and their stats."""
        lines = []
        for dsl, idx in sorted(self.dsl_to_id.items(), key=lambda x: x[1]):
            sig = self.signatures[idx]
            lines.append(f"[{idx}] {dsl:12s} n={sig.n:3d} success={sig.mean:.0%} var={sig.variance:.2f}")
        return "\n".join(lines)


# Test it
tree = SignatureTree(embedding_dim=256)
print(f"Initialized tree with {tree.num_active} signatures")

# Test deduplication
emb1 = torch.randn(256)
idx1, created1 = tree.add_signature(emb1, "a + b", "addition")
print(f"Added 'a + b': idx={idx1}, created={created1}")

emb2 = torch.randn(256)
idx2, created2 = tree.add_signature(emb2, "a + b", "addition again")
print(f"Added 'a + b' again: idx={idx2}, created={created2}")

emb3 = torch.randn(256)
idx3, created3 = tree.add_signature(emb3, "a - b", "subtraction")
print(f"Added 'a - b': idx={idx3}, created={created3}")

print(f"\nTree now has {tree.num_active} signatures (deduplicated)")

## 2. Tree-Bound LLM Model

In [ ]:
class TreeCrossAttention(nn.Module):
    """Cross-attention layer from LLM hidden states to tree signatures."""
    
    def __init__(self, hidden_size: int, signature_dim: int, num_heads: int = 4):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.signature_dim = signature_dim
        self.num_heads = num_heads
        self.head_dim = signature_dim // num_heads
        
        # Project LLM hidden → query
        self.q_proj = nn.Linear(hidden_size, signature_dim)
        
        # Project signature embeddings → key
        self.k_proj = nn.Linear(signature_dim, signature_dim)
        
        # Output projection
        self.out_proj = nn.Linear(signature_dim, signature_dim)
        
        # Layer norm
        self.norm = nn.LayerNorm(signature_dim)
        
    def forward(
        self, 
        hidden_states: torch.Tensor,  # [batch, hidden_size]
        signature_embeddings: torch.Tensor,  # [num_sigs, sig_dim]
        welford_variances: torch.Tensor,  # [num_sigs]
        return_weights: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Cross-attention to tree."""
        batch_size = hidden_states.shape[0]
        num_sigs = signature_embeddings.shape[0]
        
        Q = self.q_proj(hidden_states)
        K = self.k_proj(signature_embeddings)
        
        scores = torch.matmul(Q, K.T) / math.sqrt(self.signature_dim)
        temperature = 1.0 + welford_variances.unsqueeze(0)
        adjusted_scores = scores / temperature
        attn_weights = F.softmax(adjusted_scores, dim=-1)
        
        return Q, attn_weights, scores


class TreeBoundLLM(nn.Module):
    """LLM with native tree binding via cross-attention."""
    
    def __init__(
        self, 
        base_model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        signature_dim: int = 256,
        freeze_llm: bool = True,
        min_signatures_before_routing: int = 5,  # Force creation until we have this many
    ):
        super().__init__()
        
        self.min_signatures_before_routing = min_signatures_before_routing
        
        print(f"Loading {base_model_name}...")
        self.llm = AutoModel.from_pretrained(base_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        if freeze_llm:
            print("Freezing LLM parameters...")
            for param in self.llm.parameters():
                param.requires_grad = False
        
        hidden_size = self.llm.config.hidden_size
        self.signature_dim = signature_dim
        
        self.tree_attention = TreeCrossAttention(
            hidden_size=hidden_size,
            signature_dim=signature_dim
        )
        
        self.base_threshold = nn.Parameter(torch.tensor(0.5))
        
        print(f"TreeBoundLLM initialized:")
        print(f"  LLM hidden size: {hidden_size}")
        print(f"  Signature dim: {signature_dim}")
        print(f"  Min signatures before routing: {min_signatures_before_routing}")
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Trainable params: {trainable:,}")
    
    def get_hidden_state(self, input_ids, attention_mask, debug=False) -> torch.Tensor:
        """Get LLM hidden state for the last token."""
        if debug:
            print(f"  [get_hidden_state] input_ids: {input_ids.shape}, dtype: {input_ids.dtype}")
            print(f"  [get_hidden_state] attention_mask: {attention_mask.shape}, dtype: {attention_mask.dtype}")
        
        outputs = self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        
        if debug:
            print(f"  [get_hidden_state] last_hidden_state: {outputs.last_hidden_state.shape}")
        
        hidden = outputs.last_hidden_state[:, -1, :]
        return hidden.float()
    
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        tree: SignatureTree,
        debug: bool = False
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, bool]:
        hidden = self.get_hidden_state(input_ids, attention_mask, debug=debug)
        
        sig_embeddings = tree.get_active_embeddings().to(hidden.device)
        variances = tree.get_variances().to(hidden.device)
        
        query, route_probs, scores = self.tree_attention(
            hidden, sig_embeddings, variances
        )
        
        # Determine if we should create a new signature
        should_create = self.should_create_signature(tree, scores, debug)
        
        return query, route_probs, scores, should_create
    
    def should_create_signature(self, tree: SignatureTree, scores: torch.Tensor, debug: bool = False) -> bool:
        """
        Decide whether to create a new signature.
        
        Cold start: Force creation until we have enough basic operations
        Normal: Create if no good match exists (low max score)
        """
        num_sigs = tree.num_active
        
        # Cold start: force creation until we have basic operations
        # (but not for every single example - use some randomness)
        if num_sigs < self.min_signatures_before_routing:
            # High probability of creation in cold start
            import random
            should = random.random() < 0.7  # 70% chance to create
            if debug:
                print(f"  [should_create] Cold start mode: {should} (num_sigs={num_sigs})")
            return should
        
        # Normal mode: check if best match is good enough
        # Use raw scores (before softmax) to check actual similarity
        max_score = scores.max().item()
        threshold = self.base_threshold.detach().item()
        
        # As tree grows, be more conservative about creation
        maturity = min(1.0, num_sigs / 50)
        adjusted_threshold = threshold + 0.3 * maturity
        
        should = max_score < adjusted_threshold
        if debug:
            print(f"  [should_create] Normal mode: {should} (max_score={max_score:.3f}, threshold={adjusted_threshold:.3f})")
        
        return should


print("Testing model creation (this will download TinyLlama)...")

In [ ]:
# Initialize model and tree
model = TreeBoundLLM(
    base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    signature_dim=256,
    freeze_llm=True
).to(device)

tree = SignatureTree(embedding_dim=256).to(device)

print(f"\nTree has {tree.num_active} signatures")

In [ ]:
# Test forward pass
test_text = "solve: John has 5 apples. Mary gives him 3 more. How many apples does John have?"
inputs = model.tokenizer(test_text, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    query, route_probs, scores, should_create = model(
        inputs["input_ids"], 
        inputs["attention_mask"],
        tree
    )

print(f"Query shape: {query.shape}")
print(f"Route probs shape: {route_probs.shape}")
print(f"Should create new signature: {should_create}")
print(f"Route probs: {route_probs}")

## 3. DSL Execution

In [ ]:
def execute_dsl(dsl: str, operands: Dict[str, float], step_results: Dict[str, float]) -> Optional[float]:
    """
    Execute a simple DSL expression.
    
    DSL format: "a + b", "a - b", "a * b", "a / b"
    Operands: {"a": 5, "b": 3} or can reference step_results
    
    For non-commutative operations (-, /), tries both orderings
    and returns the one that yields a valid result.
    """
    if dsl == "DONE":
        # Return the last step result
        if step_results:
            return list(step_results.values())[-1]
        return None
    
    if dsl == "UNKNOWN":
        return None
    
    try:
        dsl = dsl.strip()
        
        # Resolve a variable name to a value
        def resolve(var):
            if var in operands:
                return float(operands[var])
            elif var in step_results:
                return float(step_results[var])
            else:
                try:
                    return float(var)  # Direct number
                except:
                    return None
        
        # Simple binary operations
        for op in ['+', '-', '*', '/']:
            if op in dsl:
                parts = dsl.split(op)
                if len(parts) == 2:
                    left = parts[0].strip()
                    right = parts[1].strip()
                    
                    a = resolve(left)
                    b = resolve(right)
                    
                    # Handle missing operands
                    if a is None or b is None:
                        return None
                    
                    # Commutative operations - straightforward
                    if op == '+':
                        return a + b
                    if op == '*':
                        return a * b
                    
                    # Non-commutative operations - try both orderings
                    # Prefer the ordering that gives a positive result
                    if op == '-':
                        result1 = a - b
                        result2 = b - a
                        # Prefer positive result, then larger operand first
                        if result1 >= 0:
                            return result1
                        elif result2 >= 0:
                            return result2
                        else:
                            # Both negative - use a - b (default ordering)
                            return result1
                    
                    if op == '/':
                        if b != 0 and a != 0:
                            # Prefer the division that gives a clean integer or >= 1
                            result1 = a / b if b != 0 else None
                            result2 = b / a if a != 0 else None
                            
                            # Check if either is a clean integer
                            if result1 is not None and result1 == int(result1):
                                return result1
                            if result2 is not None and result2 == int(result2):
                                return result2
                            
                            # Prefer result >= 1
                            if result1 is not None and result1 >= 1:
                                return result1
                            if result2 is not None and result2 >= 1:
                                return result2
                            
                            # Default to a / b
                            return result1 if result1 is not None else result2
                        elif b != 0:
                            return a / b
                        elif a != 0:
                            return b / a
                        else:
                            return None
        
        # Single value
        if dsl in operands:
            return float(operands[dsl])
        if dsl in step_results:
            return float(step_results[dsl])
            
        try:
            return float(dsl)
        except:
            return None
        
    except Exception as e:
        return None


# Test - now with smart operand handling
print("Testing DSL execution:")
print(f"  a + b with {{a:5, b:3}}: {execute_dsl('a + b', {'a': 5, 'b': 3}, {})}")
print(f"  a - b with {{a:5, b:3}}: {execute_dsl('a - b', {'a': 5, 'b': 3}, {})} (5-3)")
print(f"  a - b with {{a:3, b:5}}: {execute_dsl('a - b', {'a': 3, 'b': 5}, {})} (flips to 5-3=2)")
print(f"  a * b with {{a:4, b:7}}: {execute_dsl('a * b', {'a': 4, 'b': 7}, {})}")
print(f"  a / b with {{a:3, b:12}}: {execute_dsl('a / b', {'a': 3, 'b': 12}, {})} (flips to 12/3=4)")
print(f"  a / b with {{a:12, b:4}}: {execute_dsl('a / b', {'a': 12, 'b': 4}, {})} (12/4=3)")
print(f"  a + b with only {{a:5}}: {execute_dsl('a + b', {'a': 5}, {})}")  # Should return None
print(f"  DONE: {execute_dsl('DONE', {}, {'step_1': 42})}")

## 4. Signature Generation (LLM creates DSL)

In [ ]:
# For now, use simple heuristics to generate DSL
# In production, this would call an LLM

def generate_dsl_for_step(step_text: str, numbers: Dict[str, float]) -> Tuple[str, str]:
    """
    Generate DSL for a step. Returns (dsl, description).
    
    This is a simple heuristic version. 
    Real version would use LLM to understand the operation.
    """
    step_lower = step_text.lower()
    
    # Detect operation from keywords
    if any(w in step_lower for w in ['add', 'plus', 'more', 'gives', 'total', 'sum', 'combined']):
        return "a + b", "addition: combine two quantities"
    
    if any(w in step_lower for w in ['subtract', 'minus', 'less', 'take away', 'remain', 'left', 'ate', 'spent']):
        return "a - b", "subtraction: remove from quantity"
    
    if any(w in step_lower for w in ['multiply', 'times', 'each', 'per', 'rate']):
        return "a * b", "multiplication: repeated addition or rate calculation"
    
    if any(w in step_lower for w in ['divide', 'split', 'share', 'per', 'each gets', 'average']):
        return "a / b", "division: split into equal parts"
    
    # Default
    return "a + b", "unknown operation: defaulting to addition"


# Test
print(generate_dsl_for_step("Mary gives John 3 more apples", {}))
print(generate_dsl_for_step("John ate 2 apples", {}))
print(generate_dsl_for_step("Split the cookies among 4 friends", {}))

## 5. Training Loop with Tree Growth

In [ ]:
def load_gsm8k_data(path: str) -> List[dict]:
    """Load GSM8K data."""
    with open(path) as f:
        return [json.loads(line) for line in f]


def extract_numbers_from_problem(problem: str) -> Dict[str, float]:
    """Extract numbers from problem text."""
    import re
    numbers = re.findall(r'\b\d+(?:\.\d+)?\b', problem)
    return {f"NUM_{i}": float(n) for i, n in enumerate(numbers)}


class GSM8KDataset(Dataset):
    def __init__(self, data: List[dict], tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Format input
        question = item.get("normalized_question", item.get("question", ""))
        text = f"solve: {question}"
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "answer": float(item.get("answer", 0)),
            "num_map": item.get("num_map", {}),
            "question": question,
            "final_prefix": item.get("final_prefix", ""),
        }

In [ ]:
def train_step(
    model: TreeBoundLLM,
    tree: SignatureTree,
    batch: dict,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    debug: bool = False
) -> Tuple[float, bool, bool]:
    """
    Single training step with potential tree growth.
    
    Returns: (loss, correct, created_new)
    """
    model.train()
    
    # Handle both batched and unbatched inputs
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    
    # Ensure 2D tensors [batch, seq_len]
    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)
        attention_mask = attention_mask.unsqueeze(0)
    
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    
    if debug:
        print(f"  DEBUG: input_ids shape: {input_ids.shape}")
        print(f"  DEBUG: attention_mask shape: {attention_mask.shape}")
    
    # Handle batched scalars (DataLoader wraps in lists)
    answer = batch["answer"]
    if isinstance(answer, torch.Tensor):
        answer = answer.item()
    elif isinstance(answer, list):
        answer = answer[0]
    
    question = batch["question"]
    if isinstance(question, list):
        question = question[0]
    
    # Forward pass with debug
    query, route_probs, scores, should_create = model(
        input_ids, attention_mask, tree, debug=debug
    )
    
    if debug:
        print(f"  DEBUG: query shape: {query.shape}")
        print(f"  DEBUG: route_probs shape: {route_probs.shape}")
        print(f"  DEBUG: should_create: {should_create}")
    
    created_new = False
    
    # Handle num_map - might be a list of dicts from DataLoader
    num_map = batch["num_map"]
    if isinstance(num_map, list):
        num_map = num_map[0]
    if isinstance(num_map, str):
        num_map = json.loads(num_map)
    
    if should_create and tree.num_active < tree.max_signatures:
        # === TREE GROWTH (with deduplication) ===
        dsl, description = generate_dsl_for_step(question, num_map)
        route_idx, was_created = tree.add_signature(
            embedding=query.squeeze(),  # Tree handles device internally
            dsl=dsl,
            description=description
        )
        
        if was_created:
            created_new = True
            print(f"  [GROWTH] Created signature {route_idx}: {dsl} - {description}")
        
        # For new or deduplicated signatures, use a small initial loss
        # (Can't use route_probs since the signature may not have existed during forward pass)
        loss = torch.tensor(0.1, requires_grad=True, device=device)
    else:
        # Route to best existing signature
        route_idx = route_probs.argmax(dim=-1).item()
        
        # Compute loss
        # Use negative log prob for gradient
        loss = -torch.log(route_probs[0, route_idx] + 1e-8)
    
    # Execute DSL
    dsl = tree.get_dsl(route_idx)
    
    # Convert num_map to simple a, b format
    operands = {}
    for i, (k, v) in enumerate(num_map.items()):
        val = v[0] if isinstance(v, list) else v
        if i == 0:
            operands["a"] = float(val)
        elif i == 1:
            operands["b"] = float(val)
    
    result = execute_dsl(dsl, operands, {})
    
    # Check correctness
    correct = False
    if result is not None:
        correct = abs(result - float(answer)) < 0.01
    
    # Update Welford stats
    tree.update_signature(route_idx, correct)
    
    # Adjust loss based on correctness
    if not created_new:
        if correct:
            # Reinforce correct route (negative loss = reward)
            loss = -torch.log(route_probs[0, route_idx] + 1e-8)
        else:
            # Small penalty for incorrect route
            loss = 0.1 * torch.log(route_probs[0, route_idx] + 1e-8)
    
    # Backprop (only if loss has gradient)
    if loss.requires_grad:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    return loss.item(), correct, created_new

In [ ]:
def train_epoch(
    model: TreeBoundLLM,
    tree: SignatureTree,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_steps: int = None
) -> dict:
    """
    Train for one epoch.
    """
    total_loss = 0
    total_correct = 0
    total_created = 0
    n_steps = 0
    
    for i, batch in enumerate(dataloader):
        if max_steps and i >= max_steps:
            break
        
        # Debug first step only
        debug = (i == 0)
        
        loss, correct, created = train_step(
            model, tree, batch, optimizer, device, debug=debug
        )
        
        total_loss += loss
        total_correct += int(correct)
        total_created += int(created)
        n_steps += 1
        
        if (i + 1) % 50 == 0:
            acc = total_correct / n_steps
            print(f"  Step {i+1}: loss={total_loss/n_steps:.4f}, acc={acc:.1%}, sigs={tree.num_active}")
    
    return {
        "loss": total_loss / n_steps,
        "accuracy": total_correct / n_steps,
        "created": total_created,
        "num_signatures": tree.num_active
    }

## 6. Run Training (Cold Start)

In [ ]:
# Load data
train_data = load_gsm8k_data("data/train.jsonl")
test_data = load_gsm8k_data("data/test.jsonl")

# Filter to 1-step problems for initial experiments
train_1step = [ex for ex in train_data if ex.get("num_steps", 1) == 1]
test_1step = [ex for ex in test_data if ex.get("num_steps", 1) == 1]

print(f"Training data: {len(train_1step)} 1-step problems")
print(f"Test data: {len(test_1step)} 1-step problems")

In [ ]:
# Create fresh model and tree
model = TreeBoundLLM(
    base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    signature_dim=256,
    freeze_llm=True
).to(device)

tree = SignatureTree(embedding_dim=256, max_signatures=100).to(device)

# Dataset and dataloader
train_dataset = GSM8KDataset(train_1step, model.tokenizer)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)  # batch=1 for tree growth

# Optimizer - only train cross-attention parameters
optimizer = torch.optim.AdamW(
    model.tree_attention.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

print(f"Ready to train!")
print(f"Initial tree: {tree.num_active} signatures")

In [ ]:
# Training loop
NUM_EPOCHS = 5
STEPS_PER_EPOCH = 200  # Limit steps for faster iteration

history = []

print("="*60)
print("COLD START TRAINING")
print("Watch the tree grow!")
print("="*60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)
    
    metrics = train_epoch(
        model, tree, train_loader, optimizer, device,
        max_steps=STEPS_PER_EPOCH
    )
    
    history.append(metrics)
    
    print(f"\nEpoch {epoch+1} complete:")
    print(f"  Loss: {metrics['loss']:.4f}")
    print(f"  Accuracy: {metrics['accuracy']:.1%}")
    print(f"  Signatures created: {metrics['created']}")
    print(f"  Total signatures: {metrics['num_signatures']}")

print("\n" + "="*60)
print("TRAINING COMPLETE")
print(f"Final tree has {tree.num_active} signatures")
print("="*60)

In [ ]:
# Visualize training
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(history) + 1)

axes[0].plot(epochs, [h['loss'] for h in history], 'b-o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

axes[1].plot(epochs, [h['accuracy'] for h in history], 'g-o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')

axes[2].plot(epochs, [h['num_signatures'] for h in history], 'r-o')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('# Signatures')
axes[2].set_title('Tree Growth')

plt.tight_layout()
plt.show()

In [ ]:
# Inspect the tree using summary()
print("="*60)
print("SIGNATURE TREE SUMMARY (Deduplicated by DSL)")
print("="*60)
print(tree.summary())

print("\n" + "="*60)
print("DETAILED SIGNATURES")
print("="*60)
for idx in range(tree.num_active):
    sig = tree.signatures[idx]
    print(f"\n[{idx}] {sig.dsl}")
    print(f"    Description: {sig.description}")
    print(f"    Usage: n={sig.n}, success={sig.mean:.1%}, variance={sig.variance:.3f}")

## 7. Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, tree, test_data, device, max_samples=100):
    """Evaluate on test set."""
    model.eval()
    
    correct = 0
    total = 0
    
    for item in test_data[:max_samples]:
        question = item.get("normalized_question", item.get("question", ""))
        answer = float(item.get("answer", 0))
        num_map = item.get("num_map", {})
        
        # Tokenize
        text = f"solve: {question}"
        inputs = model.tokenizer(text, return_tensors="pt", padding=True).to(device)
        
        # Forward
        query, route_probs, scores, _ = model(
            inputs["input_ids"],
            inputs["attention_mask"],
            tree
        )
        
        # Route
        route_idx = route_probs.argmax(dim=-1).item()
        dsl = tree.get_dsl(route_idx)
        
        # Execute
        operands = {}
        for i, (k, v) in enumerate(num_map.items()):
            if i == 0:
                operands["a"] = float(v)
            elif i == 1:
                operands["b"] = float(v)
        
        result = execute_dsl(dsl, operands, {})
        
        if result is not None and abs(result - answer) < 0.01:
            correct += 1
        
        total += 1
    
    return correct / total if total > 0 else 0


# Run evaluation
test_acc = evaluate(model, tree, test_1step, device)
print(f"\nTest accuracy: {test_acc:.1%}")

In [ ]:
# Save model and tree
import pickle

torch.save({
    "tree_attention": model.tree_attention.state_dict(),
    "base_threshold": model.base_threshold,
}, "tree_bound_llm.pt")

with open("signature_tree.pkl", "wb") as f:
    pickle.dump({
        "signatures": tree.signatures,
        "embeddings": tree.embeddings.cpu(),
        "active_mask": tree.active_mask.cpu(),
        "num_active": tree.num_active,
    }, f)

print("Saved model and tree!")

# Download
files.download("tree_bound_llm.pt")
files.download("signature_tree.pkl")

## 8. Interactive Testing

In [ ]:
@torch.no_grad()
def solve_problem(model, tree, problem, device):
    """Solve a single problem."""
    model.eval()
    
    # Extract numbers
    numbers = extract_numbers_from_problem(problem)
    print(f"Numbers found: {numbers}")
    
    # Tokenize
    text = f"solve: {problem}"
    inputs = model.tokenizer(text, return_tensors="pt", padding=True).to(device)
    
    # Forward
    query, route_probs, scores, should_create = model(
        inputs["input_ids"],
        inputs["attention_mask"],
        tree
    )
    
    # Show routing
    print(f"\nRouting probabilities:")
    for idx in range(tree.num_active):
        prob = route_probs[0, idx].item()
        sig = tree.signatures[idx]
        print(f"  [{idx}] {sig.dsl:10s} : {prob:.1%}")
    
    # Route
    route_idx = route_probs.argmax(dim=-1).item()
    dsl = tree.get_dsl(route_idx)
    print(f"\nSelected: Signature {route_idx} ({dsl})")
    
    # Execute
    operands = {"a": list(numbers.values())[0] if numbers else 0,
                "b": list(numbers.values())[1] if len(numbers) > 1 else 0}
    result = execute_dsl(dsl, operands, {})
    print(f"Result: {result}")
    
    return result


# Try it!
solve_problem(model, tree, "John has 5 apples. Mary gives him 3 more. How many apples?", device)

In [ ]:
# Try more problems
problems = [
    "There are 12 cookies. 4 kids share them equally. How many does each get?",
    "A book costs 15 dollars. You have 8 dollars. How much more do you need?",
    "There are 6 rows with 7 chairs each. How many chairs total?",
]

for p in problems:
    print("\n" + "="*60)
    print(f"Problem: {p}")
    solve_problem(model, tree, p, device)